# 05 — Full Dose Projection Workflow

End-to-end example: from in vitro data and rodent PK to projected animal and human doses.

**Pipeline:**
1. Load in vitro and PK data
2. IVIVE: predict human hepatic clearance
3. Allometric scaling: predict human CL and Vss
4. Project animal efficacious doses
5. Project human dose and check absorption feasibility

In [ ]:
import sys
sys.path.insert(0, '..')

from doseprojection.io import load_invitro_data, load_pk_data
from doseprojection.ivive import ivive_workflow
from doseprojection.allometry import predict_human_cl, predict_human_vss, predict_human_thalf
from doseprojection.dose_projection import efficacious_dose_mg_kg, steady_state_css, unbound_concentration
from doseprojection.human_dose import hed_from_noael
from doseprojection.absorption import dose_number, classify_bcs, classify_permeability
import pandas as pd

## 1. Load Data

In [ ]:
invitro = load_invitro_data('../examples/example_invitro_data.csv')
pk = load_pk_data('../examples/example_pk_data.csv')

# Focus on lead compound CPD-001
cpd = invitro[invitro['compound_id'] == 'CPD-001'].iloc[0]
cpd_pk = pk[(pk['compound_id'] == 'CPD-001')]

print(f"Compound: {cpd['compound_id']}")
print(f"Target: {cpd['target']}")
print(f"IC50: {cpd['IC50_nM']} nM")
print(f"MW: {cpd['MW']}")
print(f"fu (human): {cpd['fu_plasma_human']}")
print(f"fu (rat): {cpd['fu_plasma_rat']}")
print(f"Microsomal t½ (human): {cpd['microsomal_t_half_min_human']} min")
print(f"Papp (Caco-2): {cpd['papp_caco2_cm_s']} cm/s")
print(f"Solubility: {cpd['solubility_ug_mL']} ug/mL")

## 2. IVIVE — Predict Human Hepatic Clearance

In [ ]:
ivive_result = ivive_workflow(
    t_half_min=cpd['microsomal_t_half_min_human'],
    fu=cpd['fu_plasma_human'],
    species='human'
)

print("=== IVIVE Results (Human) ===")
for k, v in ivive_result.items():
    print(f"  {k}: {v:.4f}")

## 3. Allometric Scaling — Predict Human PK

In [ ]:
# Get measured CL and Vss from rat and mouse IV data
rat_iv = cpd_pk[(cpd_pk['species'] == 'rat') & (cpd_pk['route'] == 'IV')].iloc[0]
mouse_iv = cpd_pk[(cpd_pk['species'] == 'mouse') & (cpd_pk['route'] == 'IV')].iloc[0]

# Multi-species allometric scaling for CL
cl_result = predict_human_cl({
    'mouse': mouse_iv['CL_mL_min_kg'],
    'rat': rat_iv['CL_mL_min_kg'],
})
print(f"Predicted human CL: {cl_result['cl_human_mL_min_kg']:.1f} mL/min/kg")
print(f"  Method: {cl_result['method']}")

# Scale Vss
vss_result = predict_human_vss({
    'mouse': mouse_iv['Vss_L_kg'],
    'rat': rat_iv['Vss_L_kg'],
})
print(f"Predicted human Vss: {vss_result['vss_human_L_kg']:.2f} L/kg")

# Predicted half-life
thalf = predict_human_thalf(cl_result['cl_human_mL_min_kg'], vss_result['vss_human_L_kg'])
print(f"Predicted human t½: {thalf:.1f} h")

## 4. Project Animal Efficacious Dose (Rat)

In [ ]:
rat_po = cpd_pk[(cpd_pk['species'] == 'rat') & (cpd_pk['route'] == 'PO')].iloc[0]

rat_dose = efficacious_dose_mg_kg(
    ic50_nm=cpd['IC50_nM'],
    mw=cpd['MW'],
    cl_mL_min_kg=rat_iv['CL_mL_min_kg'],
    fu=cpd['fu_plasma_rat'],
    tau_h=24,
    f=rat_po['F_pct'] / 100,
    coverage_multiple=1.0,
    route="po"
)
print(f"\n=== Rat Dose Projection (PO, QD, 1x IC50) ===")
print(f"Projected rat dose: {rat_dose:.1f} mg/kg")

css = steady_state_css(rat_dose, rat_iv['CL_mL_min_kg'], 24,
                       f=rat_po['F_pct']/100, route="po")
cu = unbound_concentration(css, cpd['fu_plasma_rat'])
print(f"Predicted Css,total: {css:.0f} ng/mL")
print(f"Predicted Cu,ss: {cu:.1f} ng/mL")

## 5. Project Human Dose

In [ ]:
# Using IVIVE-predicted human CL and allometric Vss
# Assume human F ~ Fa * Fh (simplified)
from doseprojection.absorption import predict_fa

fa = predict_fa(cpd['papp_caco2_cm_s'])
fh = ivive_result['fh']
f_human_est = fa * fh  # Simplified; ignoring Fg

# Use IVIVE CLh as human CL estimate
human_dose = efficacious_dose_mg_kg(
    ic50_nm=cpd['IC50_nM'],
    mw=cpd['MW'],
    cl_mL_min_kg=ivive_result['cl_hepatic_mL_min_kg'],
    fu=cpd['fu_plasma_human'],
    tau_h=24,
    f=f_human_est,
    coverage_multiple=1.0,
    route="po"
)

print(f"\n=== Human Dose Projection (PO, QD, 1x IC50) ===")
print(f"Estimated Fa: {fa:.2f}")
print(f"Estimated Fh: {fh:.3f}")
print(f"Estimated F (human): {f_human_est:.2f}")
print(f"IVIVE CLh (human): {ivive_result['cl_hepatic_mL_min_kg']:.2f} mL/min/kg")
print(f"Projected human dose: {human_dose:.2f} mg/kg")
print(f"Total dose (60 kg): {human_dose * 60:.0f} mg")

## 6. Absorption Feasibility Check

In [ ]:
total_dose_mg = human_dose * 60

d0 = dose_number(total_dose_mg, cpd['solubility_ug_mL'] / 1000)  # convert ug/mL to mg/mL
bcs = classify_bcs(cpd['solubility_ug_mL'] / 1000, cpd['papp_caco2_cm_s'], dose_mg=total_dose_mg)
perm = classify_permeability(cpd['papp_caco2_cm_s'])

print(f"\n=== Absorption Assessment ===")
print(f"Projected total dose: {total_dose_mg:.0f} mg")
print(f"Solubility: {cpd['solubility_ug_mL']} ug/mL = {cpd['solubility_ug_mL']/1000:.3f} mg/mL")
print(f"Dose number (D0): {d0['dose_number']:.1f}")
print(f"  → {d0['interpretation']}")
print(f"Permeability: {perm} (Papp = {cpd['papp_caco2_cm_s']:.1e} cm/s)")
print(f"BCS Class: {bcs['bcs_class']}")
print(f"  → {bcs['interpretation']}")

## 7. Summary Table

In [ ]:
summary = pd.DataFrame([
    {'Parameter': 'IC50', 'Value': f"{cpd['IC50_nM']} nM"},
    {'Parameter': 'MW', 'Value': f"{cpd['MW']}"},
    {'Parameter': 'fu (rat)', 'Value': f"{cpd['fu_plasma_rat']}"},
    {'Parameter': 'fu (human)', 'Value': f"{cpd['fu_plasma_human']}"},
    {'Parameter': 'Rat CL', 'Value': f"{rat_iv['CL_mL_min_kg']} mL/min/kg"},
    {'Parameter': 'Rat F', 'Value': f"{rat_po['F_pct']}%"},
    {'Parameter': 'Human CLh (IVIVE)', 'Value': f"{ivive_result['cl_hepatic_mL_min_kg']:.2f} mL/min/kg"},
    {'Parameter': 'Human CL (allometry)', 'Value': f"{cl_result['cl_human_mL_min_kg']:.1f} mL/min/kg"},
    {'Parameter': 'Human Vss (allometry)', 'Value': f"{vss_result['vss_human_L_kg']:.2f} L/kg"},
    {'Parameter': 'Human t½', 'Value': f"{thalf:.1f} h"},
    {'Parameter': 'Rat efficacious dose', 'Value': f"{rat_dose:.1f} mg/kg QD"},
    {'Parameter': 'Human efficacious dose', 'Value': f"{human_dose:.2f} mg/kg ({human_dose*60:.0f} mg QD)"},
    {'Parameter': 'BCS Class', 'Value': bcs['bcs_class']},
    {'Parameter': 'Dose Number', 'Value': f"{d0['dose_number']:.1f}"},
])
summary

## 8. Route Comparison: PO vs IV vs SC

For IV dosing, F=1.0 (complete bioavailability, no absorption barrier).
For SC dosing, F is the SC-specific bioavailability (no GI absorption involved).
Absorption assessment (MAD, BCS, dose number) is only relevant for oral dosing.

In [ ]:
from doseprojection.dose_projection import project_animal_dose

# --- IV dose (F=1.0 automatically) ---
iv_result = project_animal_dose(
    ic50_nm=cpd['IC50_nM'], mw=cpd['MW'],
    cl_animal_mL_min_kg=rat_iv['CL_mL_min_kg'],
    fu_animal=cpd['fu_plasma_rat'], tau_h=24,
    route="iv"  # No F needed — set to 1.0 automatically
)

# --- SC dose (requires SC-specific bioavailability) ---
f_sc = 0.65  # Example: 65% SC bioavailability
sc_result = project_animal_dose(
    ic50_nm=cpd['IC50_nM'], mw=cpd['MW'],
    cl_animal_mL_min_kg=rat_iv['CL_mL_min_kg'],
    fu_animal=cpd['fu_plasma_rat'], tau_h=24,
    f_animal=f_sc, route="sc"
)

# --- PO dose (from earlier) ---
po_result = project_animal_dose(
    ic50_nm=cpd['IC50_nM'], mw=cpd['MW'],
    cl_animal_mL_min_kg=rat_iv['CL_mL_min_kg'],
    fu_animal=cpd['fu_plasma_rat'], tau_h=24,
    f_animal=rat_po['F_pct'] / 100, route="po"
)

# Compare across routes
route_comparison = pd.DataFrame([
    {'Route': 'IV',  'F': 1.0,  'Dose (mg/kg)': round(iv_result['dose_mg_kg'], 2),
     'Css,total (ng/mL)': round(iv_result['predicted_css_total_ng_mL'], 0),
     'Note': 'F=1.0 (complete bioavailability)'},
    {'Route': 'SC',  'F': f_sc, 'Dose (mg/kg)': round(sc_result['dose_mg_kg'], 2),
     'Css,total (ng/mL)': round(sc_result['predicted_css_total_ng_mL'], 0),
     'Note': f'F_sc={f_sc} (SC bioavailability)'},
    {'Route': 'PO',  'F': rat_po['F_pct']/100, 'Dose (mg/kg)': round(po_result['dose_mg_kg'], 2),
     'Css,total (ng/mL)': round(po_result['predicted_css_total_ng_mL'], 0),
     'Note': f"F_oral={rat_po['F_pct']/100} (oral bioavailability)"},
])

print("=== Route Comparison (Rat, QD, 1x IC50 coverage) ===")
print("IV requires lowest dose (no absorption loss)")
print("Absorption assessment (MAD/BCS/dose number) only applies to PO\n")
route_comparison